In [1]:
import pandas as pd
df = pd.read_csv('rfm_segmentos.csv')
print(df.head())
print(f"\nTotal de clientes: {len(df)}")

   CustomerID  Recencia  Frecuencia  Monetario  R  F  M  Segmento  \
0        1000       276           1   24956.47  3  1  4       314   
1        1005       247           1   12349.56  3  1  3       313   
2        1012       117           1    6882.25  4  1  2       412   
3        1016       216           1   23358.77  3  1  4       314   
4        1019       331           1   31958.39  3  1  4       314   

  Nombre_Segmento  
0         Regular  
1         Regular  
2           Nuevo  
3         Regular  
4         Regular  

Total de clientes: 1802


In [2]:
# ============================================
# PREDICCIÓN DE CHURN (abandono de clientes)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Cargar datos
df = pd.read_csv('rfm_segmentos.csv')

# 2. Crear la variable objetivo (churn)
# Definimos "churn" como: clientes Perdidos o En riesgo
df['Churn'] = df['Nombre_Segmento'].apply(lambda x: 1 if x in ['Perdido', 'En riesgo'] else 0)

# 3. Seleccionar las características (features)
X = df[['Recencia', 'Frecuencia', 'Monetario']]
y = df['Churn']

# 4. Dividir en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Entrenar el modelo
modelo = RandomForestClassifier(n_estimators=100, random_state=42)
modelo.fit(X_train, y_train)

# 6. Hacer predicciones
y_pred = modelo.predict(X_test)

# 7. Evaluar el modelo
print(f"Precisión del modelo: {accuracy_score(y_test, y_pred):.2%}")
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

Precisión del modelo: 86.43%

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.91      0.87      0.89       230
           1       0.79      0.85      0.82       131

    accuracy                           0.86       361
   macro avg       0.85      0.86      0.86       361
weighted avg       0.87      0.86      0.87       361



In [4]:
# ============================================
# PREDICCIÓN PARA TODOS LOS CLIENTES
# ============================================

# Predecir churn para TODOS los clientes
df['Prediccion_Churn'] = modelo.predict(X)
df['Probabilidad_Churn'] = modelo.predict_proba(X)[:, 1]

# Clasificar en categorías de riesgo
def nivel_riesgo(prob):
    if prob >= 0.7:
        return 'Alto'
    elif prob >= 0.4:
        return 'Medio'
    else:
        return 'Bajo'

df['Nivel_Riesgo'] = df['Probabilidad_Churn'].apply(nivel_riesgo)

# Ver cuántos clientes hay en cada nivel
print("Clientes por nivel de riesgo:")
print(df['Nivel_Riesgo'].value_counts())

# Ver los 10 clientes con mayor probabilidad de churn
print("\nTop 10 clientes con mayor riesgo de abandono:")
print(df.nlargest(10, 'Probabilidad_Churn')[['CustomerID', 'Recencia', 'Frecuencia', 'Monetario', 'Probabilidad_Churn', 'Nivel_Riesgo']])

Clientes por nivel de riesgo:
Nivel_Riesgo
Bajo     1105
Alto      652
Medio      45
Name: count, dtype: int64

Top 10 clientes con mayor riesgo de abandono:
    CustomerID  Recencia  Frecuencia  Monetario  Probabilidad_Churn  \
6         1037       532           1  29163.290                 1.0   
9         1047       668           1  16548.740                 1.0   
13        1060       595           1  24420.480                 1.0   
14        1061       373           1  34165.729                 1.0   
25        1109       552           1  21546.110                 1.0   
34        1160       532           1  23018.520                 1.0   
37        1176       437           1  28647.112                 1.0   
38        1178       562           1  13665.960                 1.0   
40        1182       384           1  26861.874                 1.0   
43        1205       515           1  31469.470                 1.0   

   Nivel_Riesgo  
6          Alto  
9          Alto  
13    

In [5]:
# ============================================
# EXPORTAR RESULTADOS
# ============================================

# Guardar el dataframe con las predicciones
df.to_csv('predicciones_churn.csv', index=False)

print("✅ Archivo exportado: predicciones_churn.csv")
print(f"Total de clientes: {len(df)}")
print(f"\nColumnas del archivo:")
print(df.columns.tolist())

# Ver un resumen
print("\nResumen de niveles de riesgo:")
print(df['Nivel_Riesgo'].value_counts())

✅ Archivo exportado: predicciones_churn.csv
Total de clientes: 1802

Columnas del archivo:
['CustomerID', 'Recencia', 'Frecuencia', 'Monetario', 'R', 'F', 'M', 'Segmento', 'Nombre_Segmento', 'Churn', 'Prediccion_Churn', 'Probabilidad_Churn', 'Nivel_Riesgo']

Resumen de niveles de riesgo:
Nivel_Riesgo
Bajo     1105
Alto      652
Medio      45
Name: count, dtype: int64
